## Оптимизация выполнения кода, векторизация, Numba

Материалы:
* Макрушин С.В. Лекция 3: Оптимизация выполнения кода, векторизация, Numba
* IPython Cookbook, Second Edition (2018), глава 4
* https://numba.pydata.org/numba-doc/latest/user/5minguide.html

## Разминка

1. Сгенерируйте массив `A` из `N=1млн` случайных целых чисел на отрезке от 0 до 1000. Пусть `B[i] = A[i] + 100`. Посчитайте среднее значение массива `B`.

In [53]:
A = np.random.randint(0, 1000, size=(1000000,))

In [66]:
def f1(A):
    acc, cnt = 0, 0
    for x in A:
        acc += (x + 100)
        cnt += 1
    return acc / cnt

def f2(A):
    acc = 0
    for x in A:
        acc += (x + 100)
    return acc / len(A)

def f3(A):
    acc = 0
    for x in A:
        acc += x
    return acc / len(A) + 100

@njit
def f4(A):
    acc, cnt = 0, 0
    for x in A:
        acc += (x + 100)
        cnt += 1
    return acc / cnt

@njit
def f5(A):
    acc = 0
    for x in A:
        acc += x
    return acc / len(A) + 100

In [59]:
%timeit f1(A)

532 ms ± 15.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [60]:
%timeit f2(A)

412 ms ± 40.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [62]:
%timeit f3(A)

167 ms ± 3.74 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [65]:
%timeit f4(A)

391 µs ± 38.5 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [68]:
%timeit f5(A)

379 µs ± 13.6 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


2. Создайте таблицу 2млн строк и с 4 столбцами, заполненными случайными числами. Добавьте столбец `key`, которые содержит элементы из множества английских букв. Выберите из таблицы подмножество строк, для которых в столбце `key` указаны первые 5 английских букв.

In [95]:
import string
N = 2_000_000
df = pd.DataFrame(np.random.randn(N, 4), columns=[f'col{i}' for i in range(4)])
df['key'] = np.random.choice(list(string.ascii_letters.lower()), N, replace=True)

In [96]:
def g1(df):
    res = pd.DataFrame()
    for letter in ['a', 'b', 'c', 'd', 'e']:
        res = pd.concat([res, df[df['key']==letter]], axis=0)
    return res

def g2(df):
    res = pd.concat([df[df['key']==letter] for letter in ['a', 'b', 'c', 'd', 'e']], axis=0)
    return res

@njit
def g3(df): # ERROR
    res = pd.DataFrame()
    for letter in ['a', 'b', 'c', 'd', 'e']:
        res = pd.concat([res, df[df['key']==letter]], axis=0)
    return res

def g4(df):
    res = df[df['key'].isin(['a', 'b', 'c', 'd', 'e'])]
    return res

In [97]:
%timeit g1(df)

1.04 s ± 26.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [98]:
%timeit g2(df)

1.01 s ± 31 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [99]:
%timeit g4(df)

152 ms ± 5.31 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Лабораторная работа 3

In [ ]:
# !pip install line_profiler

In [41]:
import numpy as np
import pandas as pd
from numba import jit, njit

%load_ext line_profiler

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


1. В файлах `recipes_short.csv` и `reviews_short.csv` находится информация об рецептах блюд и отзывах на эти рецепты соответственно. Загрузите данные из файлов в виде `pd.DataFrame` с названиями `recipes` и `reviews`. Обратите внимание на корректное считывание столбца(ов) с индексами. Приведите столбцы к нужным типам.

In [2]:
recipes = pd.read_csv(r"E:\Downloads\TOBD_2021_datasets\food_com\data\recipes_short.csv", parse_dates=['submitted'])
reviews = pd.read_csv(r"E:\Downloads\TOBD_2021_datasets\food_com\data\reviews_short.csv", index_col=0, parse_dates=['date'])

In [14]:
recipes.head(2)

,name,id,minutes,contributor_id,submitted,n_steps,description,n_ingredients
0,george s at the cove black bean soup,44123,90,35193,2002-10-25,NaN,an original recipe created by chef scott meska...,18.0
1,healthy for them yogurt popsicles,67664,10,91970,2003-07-26,NaN,my children and their friends ask for my homem...,NaN


In [15]:
reviews.head(2)

,user_id,recipe_id,date,rating,review
370476,21752,57993,2003-05-01,5,Last week whole sides of frozen salmon fillet ...
624300,431813,142201,2007-09-16,5,So simple and so tasty! I used a yellow capsi...


2. Реализуйте несколько вариантов функции подсчета среднего значения столбца `rating` из таблицы `reviews` для отзывов, оставленных в 2010 году.
    1. С использованием метода `DataFrame.iterrows` и без использования срезов таблицы
    2. С использованием метода `DataFrame.iterrows` и с использованием срезов таблицы
    3. С использованием метода `DataFrame.mean`

Проверьте, что результаты работы всех написанных функций корректны и совпадают. Измерьте выполнения всех вариантов.

In [20]:
def get_mean_rating_of_2010(df):
    acc, cnt = 0, 0
    for index, row in df.iterrows():
        if row['date'].year == 2010:
            acc += row['rating']
            cnt += 1
    return acc / cnt

def get_mean_rating_of_2010_v2(df):
    acc, cnt = 0, 0
    df = df[df['date'].dt.year == 2010]
    for index, row in df.iterrows():
        acc += row['rating']
        cnt += 1
    return acc / cnt

def get_mean_rating_of_2010_v3(df):
    rating = df[df['date'].dt.year == 2010]['rating']
    acc = sum(df['rating'])
    return acc / len(rating)

def get_mean_rating_of_2010_v4(df):
    return df[df['date'].dt.year == 2010]['rating'].mean()

In [21]:
%timeit -r 2 get_mean_rating_of_2010(reviews) 

20.7 s ± 97 ms per loop (mean ± std. dev. of 2 runs, 1 loop each)


In [22]:
%timeit -r 2 get_mean_rating_of_2010_v2(reviews) 

1.97 s ± 9.27 ms per loop (mean ± std. dev. of 2 runs, 1 loop each)


In [23]:
%timeit -r 2 get_mean_rating_of_2010_v3(reviews) 

33 ms ± 6.1 µs per loop (mean ± std. dev. of 2 runs, 10 loops each)


In [25]:
%timeit -r 2 get_mean_rating_of_2010_v4(reviews) 

17.2 ms ± 94.2 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


3. Какая из созданных функций выполняется медленнее? Что наиболее сильно влияет на скорость выполнения? Для ответа использовать профайлер `line_profiler`. Сохраните результаты работы профайлера в отдельную текстовую ячейку и прокомментируйте результаты его работы.

(*). Сможете ли вы ускорить работу функции 2В?

In [28]:
%lprun -f get_mean_rating_of_2010 get_mean_rating_of_2010(reviews) 

```
Total time: 48.2732 s
File: <ipython-input-20-1027913cd611>
Function: get_mean_rating_of_2010 at line 2

Line #      Hits         Time  Per Hit   % Time  Line Contents
==============================================================
     2                                           def get_mean_rating_of_2010(df):
     3         1         21.0     21.0      0.0      acc, cnt = 0, 0
     4    126697  442715768.0   3494.3     91.7      for index, row in df.iterrows():
     5    126696   37306789.0    294.5      7.7          if row['date'].year == 2010:
     6     12094    2573898.0    212.8      0.5              acc += row['rating']
     7     12094     135048.0     11.2      0.0              cnt += 1
     8         1         15.0     15.0      0.0      return acc / cnt
```

In [29]:
%lprun -f get_mean_rating_of_2010_v2 get_mean_rating_of_2010_v2(reviews) 

```
Total time: 4.61255 s
File: <ipython-input-20-1027913cd611>
Function: get_mean_rating_of_2010_v2 at line 10

Line #      Hits         Time  Per Hit   % Time  Line Contents
==============================================================
    10                                           def get_mean_rating_of_2010_v2(df):
    11         1         22.0     22.0      0.0      acc, cnt = 0, 0
    12         1     213800.0 213800.0      0.5      df = df[df['date'].dt.year == 2010]
    13     12095   42209311.0   3489.8     91.5      for index, row in df.iterrows():
    14     12094    3550739.0    293.6      7.7          acc += row['rating']
    15     12094     151627.0     12.5      0.3          cnt += 1
    16         1         17.0     17.0      0.0      return acc / cnt
```

In [30]:
%lprun -f get_mean_rating_of_2010_v3 get_mean_rating_of_2010_v3(reviews) 

```
Total time: 0.0386734 s
File: <ipython-input-20-1027913cd611>
Function: get_mean_rating_of_2010_v3 at line 18

Line #      Hits         Time  Per Hit   % Time  Line Contents
==============================================================
    18                                           def get_mean_rating_of_2010_v3(df):
    19         1     214540.0 214540.0     55.5      rating = df[df['date'].dt.year == 2010]['rating']
    20         1     171932.0 171932.0     44.5      acc = sum(df['rating'])
    21         1        262.0    262.0      0.1      return acc / len(rating)
```

In [31]:
%lprun -f get_mean_rating_of_2010_v4 get_mean_rating_of_2010_v4(reviews) 

4. Вам предлагается воспользоваться функцией, которая собирает информацию, сколько отзывов содержат то или иное слово. Измерьте время выполнения этой функции. Сможете ли вы найти узкие места в коде, используя профайлер? Выпишите (словами), что в имеющемся коде реализовано неоптимально. Оптимизируйте функцию и добейтесь значительного прироста в скорости выполнения.

In [10]:
def get_word_reviews_count(df):
    word_reviews = {}
    for _, row in df.dropna(subset=['review']).iterrows():
        recipe_id, review = row['recipe_id'], row['review']
        words = review.split(' ')
        for word in words:
            if word not in word_reviews:
                word_reviews[word] = []
            word_reviews[word].append(recipe_id)
    
    word_reviews_count = {}
    for _, row in df.dropna(subset=['review']).iterrows():
        review = row['review']
        words = review.split(' ')
        for word in words:
            word_reviews_count[word] = len(word_reviews[word])
    return word_reviews_count

In [11]:
def get_word_reviews_count_v2(df):
    from collections import defaultdict
    word_reviews_count = defaultdict(int)
    for _, row in df.dropna(subset=['review']).iterrows():
        review = row['review']
        words = review.split(' ')
        for word in row['review'].split(' '):
            word_reviews_count[word] += 1
    return word_reviews_count

In [17]:
def get_word_reviews_count_v3(df):
    from collections import Counter
    words = reviews['review'].dropna().str.split(' ').explode()
    return Counter(words)

In [52]:
%%time
get_word_reviews_count(reviews);

Wall time: 58 s


{'Last': 94,
 'week': 804,
 'whole': 5628,
 'sides': 312,
 'of': 109029,
 'frozen': 2647,
 'salmon': 729,
 'fillet': 60,
 'was': 88781,
 'on': 34583,
 'sale': 149,
 'in': 61539,
 'my': 44144,
 'local': 561,
 'supermarket,': 10,
 'so': 46090,
 'I': 285147,
 'bought': 1369,
 'tons': 161,
 '(okay,': 5,
 'only': 13965,
 '3,': 48,
 'but': 42513,
 'total': 381,
 'weight': 160,
 'over': 9065,
 '10': 2303,
 'pounds).': 2,
 '': 214145,
 'This': 39448,
 'recipe': 41098,
 'is': 55075,
 'perfect': 4398,
 'for': 121224,
 'fillet,': 14,
 'even': 7878,
 'though': 2314,
 'it': 111175,
 'calls': 520,
 'steaks.': 93,
 'cut': 6688,
 'up': 13585,
 'the': 266050,
 'into': 7031,
 'individual': 314,
 'portions': 156,
 'and': 217849,
 'followed': 4859,
 'instructions': 731,
 'exactly.': 571,
 "I'm": 7145,
 'one': 15086,
 'those': 2287,
 'food': 2413,
 'combining': 74,
 'diets,': 5,
 'left': 4690,
 'out': 23644,
 'white': 3425,
 'wine': 1256,
 'added': 21710,
 'just': 24944,
 'a': 166136,
 'dash': 532,
 'vineg

In [12]:
%%time
get_word_reviews_count_v2(reviews);

Wall time: 31.1 s


defaultdict(int,
            {'Last': 94,
             'week': 804,
             'whole': 5628,
             'sides': 312,
             'of': 109029,
             'frozen': 2647,
             'salmon': 729,
             'fillet': 60,
             'was': 88781,
             'on': 34583,
             'sale': 149,
             'in': 61539,
             'my': 44144,
             'local': 561,
             'supermarket,': 10,
             'so': 46090,
             'I': 285147,
             'bought': 1369,
             'tons': 161,
             '(okay,': 5,
             'only': 13965,
             '3,': 48,
             'but': 42513,
             'total': 381,
             'weight': 160,
             'over': 9065,
             '10': 2303,
             'pounds).': 2,
             '': 214145,
             'This': 39448,
             'recipe': 41098,
             'is': 55075,
             'perfect': 4398,
             'for': 121224,
             'fillet,': 14,
             'even': 7878,
       

In [18]:
%%time
get_word_reviews_count_v3(reviews);

Wall time: 4.51 s


Counter({'Last': 94,
         'week': 804,
         'whole': 5628,
         'sides': 312,
         'of': 109029,
         'frozen': 2647,
         'salmon': 729,
         'fillet': 60,
         'was': 88781,
         'on': 34583,
         'sale': 149,
         'in': 61539,
         'my': 44144,
         'local': 561,
         'supermarket,': 10,
         'so': 46090,
         'I': 285147,
         'bought': 1369,
         'tons': 161,
         '(okay,': 5,
         'only': 13965,
         '3,': 48,
         'but': 42513,
         'total': 381,
         'weight': 160,
         'over': 9065,
         '10': 2303,
         'pounds).': 2,
         '': 214145,
         'This': 39448,
         'recipe': 41098,
         'is': 55075,
         'perfect': 4398,
         'for': 121224,
         'fillet,': 14,
         'even': 7878,
         'though': 2314,
         'it': 111175,
         'calls': 520,
         'steaks.': 93,
         'cut': 6688,
         'up': 13585,
         'the': 266050,
     

In [19]:
reviews

,user_id,recipe_id,date,rating,review
370476,21752,57993,2003-05-01,5,Last week whole sides of frozen salmon fillet ...
624300,431813,142201,2007-09-16,5,So simple and so tasty! I used a yellow capsi...
187037,400708,252013,2008-01-10,4,"Very nice breakfast HH, easy to make and yummy..."
706134,2001852463,404716,2017-12-11,5,These are a favorite for the holidays and so e...
312179,95810,129396,2008-03-14,5,Excellent soup! The tomato flavor is just gre...
...,...,...,...,...,...
1013457,1270706,335534,2009-05-17,4,This recipe was great! I made it last night. I...
158736,2282344,8701,2012-06-03,0,This recipe is outstanding. I followed the rec...
1059834,689540,222001,2008-04-08,5,"Well, we were not a crowd but it was a fabulou..."
453285,2000242659,354979,2015-06-02,5,I have been a steak eater and dedicated BBQ gr...


5. Напишите несколько версий функции `MAPE` (см. [MAPE](https://en.wikipedia.org/wiki/Mean_absolute_percentage_error)) для расчета среднего процентного отклонения значения рейтинга для отзыва от среднего значения рейтинга для этого отзыва. 
    1. Без использования массивов `numpy` и `numba`
    2. Без использования массивов `numpy`, но с использованием `numba`
    3. С использованием массивов `numpy`, но без использования `numba`
    4. C использованием массивов `numpy` и `numba`
    
Измерьте время выполнения каждой из реализаций.

Замечание: удалите из выборки отзывы с нулевым рейтингом.


In [30]:
mean_rating  = reviews.groupby('recipe_id')['rating'].mean().rename('mean_rating')
reviews_with_mean = reviews.merge(mean_rating, left_on='recipe_id', right_index=True)
reviews_with_mean = reviews_with_mean.query("rating > 0")

In [45]:
def mape_v1(A, F):
    acc = 0
    for idx in range(len(A)):
        acc += abs(A[idx] - F[idx]) / A[idx]
    return acc / len(A) * 100

def mape_v2(A, F):
    return np.mean(np.abs(A - F) / A) * 100

@njit
def mape_v3(A, F):
    return np.mean(np.abs(A - F) / A) * 100

@njit
def mape_v4(A, F):
    acc = 0
    for idx in range(len(A)):
        acc += abs(A[idx] - F[idx]) / A[idx]
    return acc / len(A) * 100

In [37]:
rating = reviews_with_mean['rating'].values
mean_rating = reviews_with_mean['mean_rating'].values

In [38]:
%%time
mape_v1(rating, mean_rating)

Wall time: 809 ms


13.325265503993922

In [49]:
%timeit mape_v2(rating, mean_rating)

930 µs ± 19.4 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [50]:
%timeit mape_v3(rating, mean_rating)

453 µs ± 30.8 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [51]:
%timeit mape_v4(rating, mean_rating)

585 µs ± 2.62 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
